# 交易資料的多類別分類

> 原文：[examples/Multiclass_classification_for_transactions.ipynb](https://github.com/openai/openai-cookbook/blob/main/examples/Multiclass_classification_for_transactions.ipynb)



## 這篇在做什麼

這本 notebook 把一份公開的交易資料，分類到事先定義好的幾個類別。依序試三種做法：只靠 prompt 的 zero-shot、用 embedding 加傳統分類器、以及 fine-tuning 一個模型，最後比較效果。

## 學完你會

- 用 prompt 讓模型對交易做 zero-shot 分類，並用「Could not classify」處理判斷不了的資料。
- 把交易欄位合併成一段文字做 embedding，再訓練傳統分類器。
- 把標註資料整理成 fine-tuning 的對話格式，訓練並評估分類模型。

## 開跑前

- 需要 OpenAI API key，embedding 和 fine-tuning 都會產生費用。
- 在 Colab 執行時，用 `!pip install` 裝套件，並在 Secrets 或環境變數設定 API key。

In [ ]:
# 環境設定：在 Colab 左側 🔑 Secrets 新增 OPENAI_API_KEY
%pip install -q -U numpy openai pandas scikit-learn transformers
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except ImportError:
    pass  # 本機執行就用自己的環境變數

# 抓原 repo 的資料檔（只下載這個資料夾）
if not os.path.exists('openai-cookbook'):
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/openai/openai-cookbook
    !cd openai-cookbook && git sparse-checkout set examples/data
os.chdir('openai-cookbook/examples')

# 交易資料的多類別分類

這本 notebook 要把一份公開的交易資料集，分類到我們預先定義好的幾個類別。這些做法可以套用到任何「把交易類資料歸到預定類別」的多類別分類情境。跑完之後，你會學到幾種處理有標註和沒標註資料集的方法。

這本 notebook 會用到的幾種做法：
- **Zero-shot 分類：** 先做 zero-shot 分類，只靠 prompt 引導，把交易放進五個命名好的類別
- **用 embedding 分類：** 接著對有標註的資料集建立 embedding，再用傳統分類模型測試它們辨識類別的效果
- **Fine-tuning 分類：** 最後用有標註的資料集訓練一個 fine-tuned 模型，看它跟 zero-shot 和 few-shot 分類比起來如何

## 環境設定

In [ ]:
%load_ext autoreload
%autoreload
%pip install openai 'openai[datalib]' 'openai[embeddings]' transformers scikit-learn matplotlib plotly pandas scipy


In [56]:
import openai
import pandas as pd
import numpy as np
import json
import os

COMPLETIONS_MODEL = "gpt-4"
os.environ["OPENAI_API_KEY"] = "<your-api-key>"
client = openai.OpenAI()

### 載入資料集

我們使用蘇格蘭國家圖書館（Library of Scotland）超過 £25k 的公開交易資料集。資料集有三個我們會用到的欄位：
- Supplier：供應商名稱
- Description：交易的文字描述
- Value：交易金額（英鎊）

**資料來源**：

https://data.nls.uk/data/organisational-data/transactions-over-25k/

In [152]:
transactions = pd.read_csv('./data/25000_spend_dataset_current.csv', encoding= 'unicode_escape')
print(f"Number of transactions: {len(transactions)}")
print(transactions.head())


Number of transactions: 359
         Date                      Supplier                 Description  \
0  21/04/2016          M & J Ballantyne Ltd       George IV Bridge Work   
1  26/04/2016                  Private Sale   Literary & Archival Items   
2  30/04/2016     City Of Edinburgh Council         Non Domestic Rates    
3  09/05/2016              Computacenter Uk                 Kelvin Hall   
4  09/05/2016  John Graham Construction Ltd  Causewayside Refurbishment   

   Transaction value (£)  
0                35098.0  
1                30000.0  
2                40800.0  
3                72835.0  
4                64361.0  


## Zero-shot 分類

我們先用簡單的 prompt，評估 base model 分類這些交易的表現。我們提供 5 個類別，再加一個「Could not classify」給它放不進去的交易。

In [154]:
zero_shot_prompt = '''You are a data expert working for the National Library of Scotland.
You are analysing all transactions over £25,000 in value and classifying them into one of five categories.
The five categories are Building Improvement, Literature & Archive, Utility Bills, Professional Services and Software/IT.
If you can't tell what it is, say Could not classify

Transaction:

Supplier: {}
Description: {}
Value: {}

The classification is:'''

def format_prompt(transaction):
    return zero_shot_prompt.format(transaction['Supplier'], transaction['Description'], transaction['Transaction value (£)'])

def classify_transaction(transaction):

    
    prompt = format_prompt(transaction)
    messages = [
        {"role": "system", "content": prompt},
    ]
    completion_response = openai.chat.completions.create(
                            messages=messages,
                            temperature=0,
                            max_tokens=5,
                            top_p=1,
                            frequency_penalty=0,
                            presence_penalty=0,
                            model=COMPLETIONS_MODEL)
    label = completion_response.choices[0].message.content.replace('\n','')
    return label


In [155]:
# Get a test transaction
transaction = transactions.iloc[0]
# Use our completion function to return a prediction
print(f"Transaction: {transaction['Supplier']} {transaction['Description']} {transaction['Transaction value (£)']}")
print(f"Classification: {classify_transaction(transaction)}")


Transaction: M & J Ballantyne Ltd George IV Bridge Work 35098.0
Classification: Building Improvement


第一次嘗試就對了，M & J Ballantyne Ltd 是營造商，他們做的工作確實屬於 Building Improvement。

把樣本數擴大到 25 筆，一樣只用簡單的 prompt 引導，看看表現如何

In [156]:
test_transactions = transactions.iloc[:25]
test_transactions['Classification'] = test_transactions.apply(lambda x: classify_transaction(x),axis=1)


/var/folders/3n/79rgh27s6l7_l91b9shw0_nr0000gp/T/ipykernel_81921/2775604370.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_transactions['Classification'] = test_transactions.apply(lambda x: classify_transaction(x),axis=1)


In [157]:
test_transactions['Classification'].value_counts()


Classification
Building Improvement    17
Literature & Archive     3
Software/IT              2
Could not classify       2
Utility Bills            1
Name: count, dtype: int64

In [158]:
test_transactions.head(25)


,Date,Supplier,Description,Transaction value (£),Classification
0,21/04/2016,M & J Ballantyne Ltd,George IV Bridge Work,35098.0,Building Improvement
1,26/04/2016,Private Sale,Literary & Archival Items,30000.0,Literature & Archive
2,30/04/2016,City Of Edinburgh Council,Non Domestic Rates,40800.0,Utility Bills
3,09/05/2016,Computacenter Uk,Kelvin Hall,72835.0,Software/IT
4,09/05/2016,John Graham Construction Ltd,Causewayside Refurbishment,64361.0,Building Improvement
5,09/05/2016,A McGillivray,Causewayside Refurbishment,53690.0,Building Improvement
6,16/05/2016,John Graham Construction Ltd,Causewayside Refurbishment,365344.0,Building Improvement
7,23/05/2016,Computacenter Uk,Kelvin Hall,26506.0,Software/IT
8,23/05/2016,ECG Facilities Service,Facilities Management Charge,32777.0,Building Improvement
9,23/05/2016,ECG Facilities Service,Facilities Management Charge,32777.0,Building Improvement


沒有任何標註範例，初步結果就相當不錯！它沒辦法分類的都是比較難的案例，線索很少，看不出主題。不過如果我們把標註資料集整理乾淨，提供更多範例，也許表現會更好。

## 用 embedding 分類

我們用目前已經分好類的小資料集來建立 embedding。這個標註範例集的做法是：對資料集中 101 筆交易跑 zero-shot 分類器，再手動修正其中 15 筆 **Could not classify** 的結果。

### 建立 embedding

這一段沿用 [Get_embeddings_from_dataset Notebook](Get_embeddings_from_dataset.ipynb) 的做法，把所有欄位串成一個合併欄位，再對它建立 embedding

In [ ]:
df = pd.read_csv('./data/labelled_transactions.csv')
df.head()

,Date,Supplier,Description,Transaction value (£),Classification
0,15/08/2016,Creative Video Productions Ltd,Kelvin Hall,26866,Other
1,29/05/2017,John Graham Construction Ltd,Causewayside Refurbishment,74806,Building Improvement
2,29/05/2017,Morris & Spottiswood Ltd,George IV Bridge Work,56448,Building Improvement
3,31/05/2017,John Graham Construction Ltd,Causewayside Refurbishment,164691,Building Improvement
4,24/07/2017,John Graham Construction Ltd,Causewayside Refurbishment,27926,Building Improvement


In [160]:
df['combined'] = "Supplier: " + df['Supplier'].str.strip() + "; Description: " + df['Description'].str.strip() + "; Value: " + str(df['Transaction value (£)']).strip()
df.head(2)

,Date,Supplier,Description,Transaction value (£),Classification,combined
0,15/08/2016,Creative Video Productions Ltd,Kelvin Hall,26866,Other,Supplier: Creative Video Productions Ltd; Desc...
1,29/05/2017,John Graham Construction Ltd,Causewayside Refurbishment,74806,Building Improvement,Supplier: John Graham Construction Ltd; Descri...


In [161]:
from transformers import GPT2TokenizerFast
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

df['n_tokens'] = df.combined.apply(lambda x: len(tokenizer.encode(x)))
len(df)

101

In [162]:
embedding_path = './data/transactions_with_embeddings_100.csv'

In [163]:
from utils.embeddings_utils import get_embedding
df['babbage_similarity'] = df.combined.apply(lambda x: get_embedding(x))
df['babbage_search'] = df.combined.apply(lambda x: get_embedding(x))
df.to_csv(embedding_path)


### 用 embedding 做分類

有了 embedding，來看看把交易分到我們命名的類別，結果會不會更好。

這裡用 [Classification_using_embeddings](Classification_using_embeddings.ipynb) notebook 的範本

In [164]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from ast import literal_eval

fs_df = pd.read_csv(embedding_path)
fs_df["babbage_similarity"] = fs_df.babbage_similarity.apply(literal_eval).apply(np.array)
fs_df.head()


,Unnamed: 0,Date,Supplier,Description,Transaction value (£),Classification,combined,n_tokens,babbage_similarity,babbage_search
0,0,15/08/2016,Creative Video Productions Ltd,Kelvin Hall,26866,Other,Supplier: Creative Video Productions Ltd; Desc...,136,"[-0.02898375503718853, -0.02881557121872902, 0...","[-0.02879939414560795, -0.02867320366203785, 0..."
1,1,29/05/2017,John Graham Construction Ltd,Causewayside Refurbishment,74806,Building Improvement,Supplier: John Graham Construction Ltd; Descri...,140,"[-0.024112487211823463, -0.02881261520087719, ...","[-0.024112487211823463, -0.02881261520087719, ..."
2,2,29/05/2017,Morris & Spottiswood Ltd,George IV Bridge Work,56448,Building Improvement,Supplier: Morris & Spottiswood Ltd; Descriptio...,141,"[0.013581369072198868, -0.003978211898356676, ...","[0.013593776151537895, -0.0037341134157031775,..."
3,3,31/05/2017,John Graham Construction Ltd,Causewayside Refurbishment,164691,Building Improvement,Supplier: John Graham Construction Ltd; Descri...,140,"[-0.024112487211823463, -0.02881261520087719, ...","[-0.024112487211823463, -0.02881261520087719, ..."
4,4,24/07/2017,John Graham Construction Ltd,Causewayside Refurbishment,27926,Building Improvement,Supplier: John Graham Construction Ltd; Descri...,140,"[-0.02408558875322342, -0.02881370671093464, 0...","[-0.024109570309519768, -0.02880912832915783, ..."


In [165]:
X_train, X_test, y_train, y_test = train_test_split(
    list(fs_df.babbage_similarity.values), fs_df.Classification, test_size=0.2, random_state=42
)

clf = RandomForestClassifier(n_estimators=100)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)
probas = clf.predict_proba(X_test)

report = classification_report(y_test, preds)
print(report)


                      precision    recall  f1-score   support

Building Improvement       0.92      1.00      0.96        11
Literature & Archive       1.00      1.00      1.00         3
               Other       0.00      0.00      0.00         1
         Software/IT       1.00      1.00      1.00         1
       Utility Bills       1.00      1.00      1.00         5

            accuracy                           0.95        21
           macro avg       0.78      0.80      0.79        21
        weighted avg       0.91      0.95      0.93        21



/Users/vishnu/code/openai-cookbook/cookbook_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/vishnu/code/openai-cookbook/cookbook_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/vishnu/code/openai-cookbook/cookbook_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

這個模型的表現相當強，所以建立 embedding 再搭配更簡單的分類器，看起來也是有效的做法，而 zero-shot 分類器幫我們完成了未標註資料集的初步分類。

再往前一步，看看用同一份標註資料集訓練的 fine-tuned 模型，能不能得到相當的結果

## Fine-tuning 交易分類

這個使用情境要改進上面的 few-shot 分類，做法是用同樣 101 筆標註交易訓練一個 fine-tuned 模型，再把它套用在一批沒看過的交易上

### 建立 fine-tuned 分類器

先做一些資料準備，步驟如下：
- 準備訓練集和驗證集時，我們會建立一組 message 序列。每筆的第一則 message 是帶入交易細節的 user prompt，最後一則 message 是模型預期的分類回應
- 測試集包含每筆交易的初始 user prompt，以及對應的預期類別標籤。之後用 fine-tuned 模型產生每筆交易的實際分類結果。

In [64]:
ft_prep_df = fs_df.copy()
len(ft_prep_df)


101

In [65]:
ft_prep_df.head()


,Unnamed: 0,Date,Supplier,Description,Transaction value (£),Classification,combined,n_tokens,babbage_similarity,babbage_search
0,0,15/08/2016,Creative Video Productions Ltd,Kelvin Hall,26866,Other,Supplier: Creative Video Productions Ltd; Desc...,136,"[-0.028885245323181152, -0.028660893440246582,...","[-0.02879939414560795, -0.02867320366203785, 0..."
1,1,29/05/2017,John Graham Construction Ltd,Causewayside Refurbishment,74806,Building Improvement,Supplier: John Graham Construction Ltd; Descri...,140,"[-0.024112487211823463, -0.02881261520087719, ...","[-0.02414606139063835, -0.02883070334792137, 0..."
2,2,29/05/2017,Morris & Spottiswood Ltd,George IV Bridge Work,56448,Building Improvement,Supplier: Morris & Spottiswood Ltd; Descriptio...,141,"[0.013593776151537895, -0.0037341134157031775,...","[0.013561442494392395, -0.004199974238872528, ..."
3,3,31/05/2017,John Graham Construction Ltd,Causewayside Refurbishment,164691,Building Improvement,Supplier: John Graham Construction Ltd; Descri...,140,"[-0.024112487211823463, -0.02881261520087719, ...","[-0.024112487211823463, -0.02881261520087719, ..."
4,4,24/07/2017,John Graham Construction Ltd,Causewayside Refurbishment,27926,Building Improvement,Supplier: John Graham Construction Ltd; Descri...,140,"[-0.024112487211823463, -0.02881261520087719, ...","[-0.024112487211823463, -0.02881261520087719, ..."


In [96]:
classes = list(set(ft_prep_df['Classification']))
class_df = pd.DataFrame(classes).reset_index()
class_df.columns = ['class_id','class']
class_df  , len(class_df)


(   class_id                 class
 0         0                 Other
 1         1  Literature & Archive
 2         2           Software/IT
 3         3         Utility Bills
 4         4  Building Improvement,
 5)

In [181]:
ft_df_with_class = ft_prep_df.merge(class_df,left_on='Classification',right_on='class',how='inner')

# Creating a list of messages for the fine-tuning job. The user message is the prompt, and the assistant message is the response from the model
ft_df_with_class['messages'] = ft_df_with_class.apply(lambda x: [{"role": "user", "content": format_prompt(x)}, {"role": "assistant", "content": x['class']}],axis=1)
ft_df_with_class[['messages', 'class']].head()


,messages,class
0,"[{'role': 'user', 'content': 'You are a data e...",Other
1,"[{'role': 'user', 'content': 'You are a data e...",Building Improvement
2,"[{'role': 'user', 'content': 'You are a data e...",Building Improvement
3,"[{'role': 'user', 'content': 'You are a data e...",Building Improvement
4,"[{'role': 'user', 'content': 'You are a data e...",Building Improvement


In [169]:
# Create train/validation split
samples = ft_df_with_class["messages"].tolist()
train_df, valid_df = train_test_split(samples, test_size=0.2, random_state=42)

def write_to_jsonl(list_of_messages, filename):
    with open(filename, "w+") as f:
        for messages in list_of_messages:
            object = {  
                "messages": messages
            }
            f.write(json.dumps(object) + "\n")


In [186]:
# Write the train/validation split to jsonl files
train_file_name, valid_file_name = "transactions_grouped_train.jsonl", "transactions_grouped_valid.jsonl"
write_to_jsonl(train_df, train_file_name)
write_to_jsonl(valid_df, valid_file_name)


In [ ]:
# Upload the files to OpenAI
train_file = client.files.create(file=open(train_file_name, "rb"), purpose="fine-tune")
valid_file = client.files.create(file=open(valid_file_name, "rb"), purpose="fine-tune")

In [ ]:
# Create the fine-tuning job
fine_tuning_job = client.fine_tuning.jobs.create(training_file=train_file.id, validation_file=valid_file.id, model="gpt-4o-2024-08-06")
# Get the fine-tuning job status and model name
status = client.fine_tuning.jobs.retrieve(fine_tuning_job.id)

In [209]:
# Once the fine-tuning job is complete, you can retrieve the model name from the job status
fine_tuned_model = client.fine_tuning.jobs.retrieve(fine_tuning_job.id).fine_tuned_model
print(f"Fine tuned model id: {fine_tuned_model}")

Fine tuned model id: ft:gpt-4o-2024-08-06:openai::BKr3Xy8U


### 套用 fine-tuned 分類器

現在套用分類器看看表現如何。訓練集只有 31 筆不重複的觀察值，驗證集 8 筆，來看看成績

In [210]:
# Create a test set with the expected class labels
test_set = pd.read_json(valid_file_name, lines=True)
test_set['expected_class'] = test_set.apply(lambda x: x['messages'][-1]['content'], axis=1)
test_set.head()

,messages,expected_class
0,"[{'role': 'user', 'content': 'You are a data e...",Utility Bills
1,"[{'role': 'user', 'content': 'You are a data e...",Literature & Archive
2,"[{'role': 'user', 'content': 'You are a data e...",Literature & Archive
3,"[{'role': 'user', 'content': 'You are a data e...",Literature & Archive
4,"[{'role': 'user', 'content': 'You are a data e...",Building Improvement


In [ ]:
# Apply the fine-tuned model to the test set
test_set['response'] = test_set.apply(lambda x: openai.chat.completions.create(model=fine_tuned_model, messages=x['messages'][:-1], temperature=0),axis=1)
test_set['predicted_class'] = test_set.apply(lambda x: x['response'].choices[0].message.content, axis=1)

test_set.head()

,messages,expected_class,response,predicted_class
0,"[{'role': 'user', 'content': 'You are a data e...",Utility Bills,ChatCompletion(id='chatcmpl-BKrC0S1wQSfM9ZQfcC...,Utility Bills
1,"[{'role': 'user', 'content': 'You are a data e...",Literature & Archive,ChatCompletion(id='chatcmpl-BKrC1BTr0DagbDkC2s...,Literature & Archive
2,"[{'role': 'user', 'content': 'You are a data e...",Literature & Archive,ChatCompletion(id='chatcmpl-BKrC1H3ZeIW5cz2Owr...,Literature & Archive
3,"[{'role': 'user', 'content': 'You are a data e...",Literature & Archive,ChatCompletion(id='chatcmpl-BKrC1wdhaMP0Q7YmYx...,Literature & Archive
4,"[{'role': 'user', 'content': 'You are a data e...",Building Improvement,ChatCompletion(id='chatcmpl-BKrC20c5pkpngy1xDu...,Building Improvement


In [212]:
# Calculate the accuracy of the predictions
from sklearn.metrics import f1_score
test_set['result'] = test_set.apply(lambda x: str(x['predicted_class']).strip() == str(x['expected_class']).strip(), axis = 1)
test_set['result'].value_counts()

print(test_set['result'].value_counts())

print("F1 Score: ", f1_score(test_set['expected_class'], test_set['predicted_class'], average="weighted"))
print("Raw Accuracy: ", test_set['result'].value_counts()[True] / len(test_set))


result
True     20
False     1
Name: count, dtype: int64
F1 Score:  0.9296066252587991
Raw Accuracy:  0.9523809523809523


## 重點整理

- 沒有標註資料時，zero-shot 加一個「Could not classify」兜底類別，就能快速做出初步分類。
- 用 zero-shot 的結果再人工修正，是低成本做出標註資料集的好辦法。
- embedding 加簡單的傳統分類器，通常就能有很好的表現，成本也低。
- Fine-tuning 要把資料整理成 user prompt 加預期答案的 message 格式，樣本很少時也能試。
- 三種做法沒有絕對的贏家，要看你有多少標註資料、能接受多少成本。

## 練習

1. 把 zero-shot 的樣本數從 25 改成 50，看「Could not classify」的比例有沒有變化。
2. 修改 zero-shot 的 prompt，增加或調整類別描述，觀察分類結果有什麼不同。
3. 把用 embedding 分類時的分類器換成另一種（例如 logistic regression），跟原本的比較準確率。